# RDMA Fundamentals

Establish a safe, read-only baseline for RDMA device and utility discovery.

## Objectives

Identify available RDMA tools and frame an experiment around devices, ports, and data movement.

## Background

RDMA enables direct data movement with low CPU involvement, but measurements depend on device, link, memory registration, queue, and topology details.

## Prediction

The DGX Spark should expose one or more Mellanox/NVIDIA RDMA-capable devices through the Linux RDMA subsystem.

Before running the experiment, I predict:

1. At least one of `ibv_devices`, `ibstat`, and `rdma` will be installed.
2. `/sys/class/infiniband` will contain one or more RDMA devices associated with the high-speed ConnectX interfaces.
3. Each physical port will report a link layer, administrative state, physical state, and nominal link rate.
4. The RDMA devices will map to Linux network interfaces, but an active physical link does not necessarily imply that an IP address or usable RDMA route is configured.
5. The reported link layer may be Ethernet rather than native InfiniBand. In that case, later RDMA experiments would use RoCE and would depend on Ethernet addressing and configuration.

These are architectural expectations, not measured properties. The experiment below records the actual local configuration.

## Environment

In [1]:
import os
import platform
import socket
import sys
from pathlib import Path

repository_root = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "pyproject.toml").is_file()
    ),
    None,
)
if repository_root is None:
    raise RuntimeError("Run this notebook from within the repository")
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

print(f"Python: {sys.version}")
print(f"Platform: {platform.platform()}")
print(f"Hostname: {socket.gethostname()}")
print(f"Working directory: {os.getcwd()}")

Python: 3.14.3 (main, Feb 12 2026, 00:45:04) [Clang 21.1.4 ]
Platform: Linux-6.17.0-1026-nvidia-aarch64-with-glibc2.39
Hostname: spark-0240
Working directory: /home/coert/workspace/dgx-spark-lab/experiments/00-foundations


## Experiment

In [2]:
from common.rdma import detect_rdma_utilities, run_rdma_utility


rdma_utilities = detect_rdma_utilities()

for utility in rdma_utilities:
    status = "available" if utility.available else "missing"
    print(f"{utility.name:<12} {status:<9} {utility.path or '-'}")

ibv_devices  available /usr/bin/ibv_devices
ibstat       missing   -
rdma         available /usr/bin/rdma


In [3]:
def print_command_result(result) -> None:
    command_text = " ".join(result.command)

    print(f"$ {command_text}")
    print(f"return code: {result.returncode}")

    if result.timed_out:
        print("status: timed out")
    elif result.executable_missing:
        print("status: executable missing")
    elif result.error is not None:
        print(f"status: {result.error}")
    else:
        print(f"status: {'succeeded' if result.succeeded else 'failed'}")

    if result.stdout.strip():
        print("\nstdout:")
        print(result.stdout.rstrip())

    if result.stderr.strip():
        print("\nstderr:")
        print(result.stderr.rstrip())

    print()


available_utility_names = {
    utility.name for utility in rdma_utilities if utility.available
}

rdma_command_results = []

if "ibv_devices" in available_utility_names:
    rdma_command_results.append(run_rdma_utility("ibv_devices"))

if "ibstat" in available_utility_names:
    rdma_command_results.append(run_rdma_utility("ibstat"))

if "rdma" in available_utility_names:
    rdma_command_results.extend(
        [
            run_rdma_utility("rdma", "dev", "show"),
            run_rdma_utility("rdma", "link", "show"),
        ]
    )

if not rdma_command_results:
    print("No supported RDMA utilities were available.")
else:
    for result in rdma_command_results:
        print_command_result(result)

$ ibv_devices
return code: 0
status: succeeded

stdout:
    device          	   node GUID
    ------          	----------------
    rocep1s0f0      	4cbb470300830241
    rocep1s0f1      	4cbb470300830242
    roceP2p1s0f0    	4cbb470300830245
    roceP2p1s0f1    	4cbb470300830246

$ rdma dev show
return code: 0
status: succeeded

stdout:
0: rocep1s0f0: node_type ca fw 28.45.4028 node_guid 4cbb:4703:0083:0241 sys_image_guid 4cbb:4703:0083:0241 
1: rocep1s0f1: node_type ca fw 28.45.4028 node_guid 4cbb:4703:0083:0242 sys_image_guid 4cbb:4703:0083:0241 
2: roceP2p1s0f0: node_type ca fw 28.45.4028 node_guid 4cbb:4703:0083:0245 sys_image_guid 4cbb:4703:0083:0241 
3: roceP2p1s0f1: node_type ca fw 28.45.4028 node_guid 4cbb:4703:0083:0246 sys_image_guid 4cbb:4703:0083:0241

$ rdma link show
return code: 0
status: succeeded

stdout:
link rocep1s0f0/1 state DOWN physical_state DISABLED netdev enp1s0f0np0 
link rocep1s0f1/1 state ACTIVE physical_state LINK_UP netdev enp1s0f1np1 
link roceP2p1s0f0/1

### Linux sysfs inventory

Command-line utilities present a user-space view of the RDMA subsystem. Linux also exposes device, port, and network-interface relationships through `/sys/class/infiniband`.

The following cell reads that hierarchy directly so that the inventory remains useful even if some optional RDMA utilities are absent.

In [4]:
import pandas as pd


def read_text(path: Path) -> str | None:
    try:
        return path.read_text().strip()
    except (FileNotFoundError, PermissionError, OSError):
        return None


infiniband_root = Path("/sys/class/infiniband")
rdma_device_rows = []
rdma_port_rows = []

if not infiniband_root.is_dir():
    print(f"{infiniband_root} does not exist.")
else:
    rdma_devices = sorted(path for path in infiniband_root.iterdir() if path.is_dir())

    if not rdma_devices:
        print(f"No RDMA devices were found under {infiniband_root}.")

    for rdma_device in rdma_devices:
        device_path = rdma_device / "device"
        network_path = device_path / "net"

        network_interfaces = (
            sorted(path.name for path in network_path.iterdir())
            if network_path.is_dir()
            else []
        )

        pci_device = None
        try:
            pci_device = device_path.resolve().name
        except OSError:
            pass

        driver = None
        driver_path = device_path / "driver"
        try:
            if driver_path.exists():
                driver = driver_path.resolve().name
        except OSError:
            pass

        rdma_device_rows.append(
            {
                "rdma_device": rdma_device.name,
                "node_type": read_text(rdma_device / "node_type"),
                "firmware_version": read_text(rdma_device / "fw_ver"),
                "node_guid": read_text(rdma_device / "node_guid"),
                "sys_image_guid": read_text(rdma_device / "sys_image_guid"),
                "pci_device": pci_device,
                "driver": driver,
                "network_interfaces": ", ".join(network_interfaces) or None,
            }
        )

        ports_path = rdma_device / "ports"
        if not ports_path.is_dir():
            continue

        for port_path in sorted(
            ports_path.iterdir(),
            key=lambda path: int(path.name) if path.name.isdigit() else path.name,
        ):
            if not port_path.is_dir():
                continue

            rdma_port_rows.append(
                {
                    "rdma_device": rdma_device.name,
                    "port": port_path.name,
                    "state": read_text(port_path / "state"),
                    "physical_state": read_text(port_path / "phys_state"),
                    "link_layer": read_text(port_path / "link_layer"),
                    "rate": read_text(port_path / "rate"),
                    "lid": read_text(port_path / "lid"),
                    "lid_mask_count": read_text(port_path / "lid_mask_count"),
                    "sm_lid": read_text(port_path / "sm_lid"),
                }
            )

rdma_devices_df = pd.DataFrame(rdma_device_rows)
rdma_ports_df = pd.DataFrame(rdma_port_rows)

print("RDMA devices")
display(rdma_devices_df)

print("\nRDMA ports")
display(rdma_ports_df)

RDMA devices


,rdma_device,node_type,firmware_version,node_guid,sys_image_guid,pci_device,driver,network_interfaces
0,roceP2p1s0f0,1: CA,28.45.4028,4cbb:4703:0083:0245,4cbb:4703:0083:0241,0002:01:00.0,mlx5_core,enP2p1s0f0np0
1,roceP2p1s0f1,1: CA,28.45.4028,4cbb:4703:0083:0246,4cbb:4703:0083:0241,0002:01:00.1,mlx5_core,enP2p1s0f1np1
2,rocep1s0f0,1: CA,28.45.4028,4cbb:4703:0083:0241,4cbb:4703:0083:0241,0000:01:00.0,mlx5_core,enp1s0f0np0
3,rocep1s0f1,1: CA,28.45.4028,4cbb:4703:0083:0242,4cbb:4703:0083:0241,0000:01:00.1,mlx5_core,enp1s0f1np1



RDMA ports


,rdma_device,port,state,physical_state,link_layer,rate,lid,lid_mask_count,sm_lid
0,roceP2p1s0f0,1,1: DOWN,3: Disabled,Ethernet,40 Gb/sec (4X QDR),0x0,0,0x0
1,roceP2p1s0f1,1,4: ACTIVE,5: LinkUp,Ethernet,200 Gb/sec (4X HDR),0x0,0,0x0
2,rocep1s0f0,1,1: DOWN,3: Disabled,Ethernet,40 Gb/sec (4X QDR),0x0,0,0x0
3,rocep1s0f1,1,4: ACTIVE,5: LinkUp,Ethernet,200 Gb/sec (4X HDR),0x0,0,0x0


### Associated network interfaces

RDMA device discovery alone does not show whether the corresponding Linux network interfaces are administratively enabled, physically connected, or assigned addresses.

The next cell reads the ordinary Linux interface state and uses `ip` only for read-only address and route reporting.

In [5]:
from common.utils import run_command


rdma_network_interfaces = sorted(
    {
        interface
        for row in rdma_device_rows
        for interface in (row["network_interfaces"] or "").split(", ")
        if interface
    }
)

network_interface_rows = []

for interface in rdma_network_interfaces:
    interface_path = Path("/sys/class/net") / interface

    network_interface_rows.append(
        {
            "interface": interface,
            "operstate": read_text(interface_path / "operstate"),
            "carrier": read_text(interface_path / "carrier"),
            "mtu": read_text(interface_path / "mtu"),
            "speed_mbps": read_text(interface_path / "speed"),
            "duplex": read_text(interface_path / "duplex"),
            "address": read_text(interface_path / "address"),
        }
    )

network_interfaces_df = pd.DataFrame(network_interface_rows)

print("RDMA-associated network interfaces")
display(network_interfaces_df)

if not rdma_network_interfaces:
    print("No Linux network interfaces were mapped to the RDMA devices.")
else:
    for interface in rdma_network_interfaces:
        print_command_result(
            run_command(("ip", "-details", "address", "show", "dev", interface))
        )

    print_command_result(run_command(("ip", "route", "show")))

RDMA-associated network interfaces


,interface,operstate,carrier,mtu,speed_mbps,duplex,address
0,enP2p1s0f0np0,down,0,1500,-1,unknown,4c:bb:47:83:02:45
1,enP2p1s0f1np1,up,1,9000,200000,full,4c:bb:47:83:02:46
2,enp1s0f0np0,down,0,1500,-1,unknown,4c:bb:47:83:02:41
3,enp1s0f1np1,up,1,9000,200000,full,4c:bb:47:83:02:42


$ ip -details address show dev enP2p1s0f0np0
return code: 0
status: succeeded

stdout:
5: enP2p1s0f0np0: <NO-CARRIER,BROADCAST,MULTICAST,UP> mtu 1500 qdisc mq state DOWN group default qlen 1000
    link/ether 4c:bb:47:83:02:45 brd ff:ff:ff:ff:ff:ff promiscuity 0  allmulti 0 minmtu 68 maxmtu 9978 numtxqueues 424 numrxqueues 20 gso_max_size 65536 gso_max_segs 65535 tso_max_size 524280 tso_max_segs 65535 gro_max_size 65536 portname p0 switchid 410283000347bb4c parentbus pci parentdev 0002:01:00.0

$ ip -details address show dev enP2p1s0f1np1
return code: 0
status: succeeded

stdout:
6: enP2p1s0f1np1: <BROADCAST,MULTICAST,UP,LOWER_UP> mtu 9000 qdisc mq state UP group default qlen 1000
    link/ether 4c:bb:47:83:02:46 brd ff:ff:ff:ff:ff:ff promiscuity 0  allmulti 0 minmtu 68 maxmtu 9978 numtxqueues 424 numrxqueues 20 gso_max_size 65536 gso_max_segs 65535 tso_max_size 524280 tso_max_segs 65535 gro_max_size 65536 portname p1 switchid 410283000347bb4c parentbus pci parentdev 0002:01:00.1 
    

### Prediction: peer reachability and benchmark support

Both active `/30` networks should contain one remote endpoint, conventionally `.2`:

- local `10.200.0.1` should reach peer `10.200.0.2` through `enP2p1s0f1np1`;
- local `10.201.0.1` should reach peer `10.201.0.2` through `enp1s0f1np1`.

Linux route lookup should select a different physical interface for each peer.

The system may also provide additional verbs and `perftest` utilities beyond the three commands currently handled by `common.rdma`. Discovering those utilities does not prove that RDMA communication works; it only determines which later experiments are locally available.

In [6]:
from shutil import which


candidate_rdma_utilities = (
    "ibv_devinfo",
    "ibv_rc_pingpong",
    "rping",
    "ib_send_lat",
    "ib_write_lat",
    "ib_read_lat",
    "ib_send_bw",
    "ib_write_bw",
    "ib_read_bw",
)

rdma_tool_rows = [
    {
        "utility": utility,
        "available": which(utility) is not None,
        "path": which(utility),
    }
    for utility in candidate_rdma_utilities
]

rdma_tools_df = pd.DataFrame(rdma_tool_rows)
display(rdma_tools_df)

,utility,available,path
0,ibv_devinfo,True,/usr/bin/ibv_devinfo
1,ibv_rc_pingpong,True,/usr/bin/ibv_rc_pingpong
2,rping,True,/usr/bin/rping
3,ib_send_lat,True,/usr/bin/ib_send_lat
4,ib_write_lat,True,/usr/bin/ib_write_lat
5,ib_read_lat,True,/usr/bin/ib_read_lat
6,ib_send_bw,True,/usr/bin/ib_send_bw
7,ib_write_bw,True,/usr/bin/ib_write_bw
8,ib_read_bw,True,/usr/bin/ib_read_bw


In [7]:
peer_paths = (
    {
        "local_address": "10.200.0.1",
        "peer_address": "10.200.0.2",
        "expected_interface": "enP2p1s0f1np1",
    },
    {
        "local_address": "10.201.0.1",
        "peer_address": "10.201.0.2",
        "expected_interface": "enp1s0f1np1",
    },
)

peer_path_results = []

for path in peer_paths:
    route_result = run_command(
        ("ip", "route", "get", path["peer_address"]),
        timeout=5.0,
    )
    ping_result = run_command(
        (
            "ping",
            "-n",
            "-c",
            "5",
            "-W",
            "1",
            "-I",
            path["expected_interface"],
            path["peer_address"],
        ),
        timeout=10.0,
    )

    peer_path_results.append(
        {
            **path,
            "route_succeeded": route_result.succeeded,
            "route_output": route_result.stdout.strip(),
            "ping_succeeded": ping_result.succeeded,
            "ping_output": ping_result.stdout.strip(),
            "ping_error": ping_result.stderr.strip() or ping_result.error,
        }
    )

    print_command_result(route_result)
    print_command_result(ping_result)

peer_paths_df = pd.DataFrame(peer_path_results)
display(
    peer_paths_df[
        [
            "local_address",
            "peer_address",
            "expected_interface",
            "route_succeeded",
            "ping_succeeded",
        ]
    ]
)

$ ip route get 10.200.0.2
return code: 0
status: succeeded

stdout:
10.200.0.2 dev enP2p1s0f1np1 src 10.200.0.1 uid 1001 
    cache

$ ping -n -c 5 -W 1 -I enP2p1s0f1np1 10.200.0.2
return code: 0
status: succeeded

stdout:
PING 10.200.0.2 (10.200.0.2) from 10.200.0.1 enP2p1s0f1np1: 56(84) bytes of data.
64 bytes from 10.200.0.2: icmp_seq=1 ttl=64 time=0.404 ms
64 bytes from 10.200.0.2: icmp_seq=2 ttl=64 time=0.749 ms
64 bytes from 10.200.0.2: icmp_seq=3 ttl=64 time=0.703 ms
64 bytes from 10.200.0.2: icmp_seq=4 ttl=64 time=0.339 ms
64 bytes from 10.200.0.2: icmp_seq=5 ttl=64 time=0.732 ms

--- 10.200.0.2 ping statistics ---
5 packets transmitted, 5 received, 0% packet loss, time 4134ms
rtt min/avg/max/mdev = 0.339/0.585/0.749/0.176 ms

$ ip route get 10.201.0.2
return code: 0
status: succeeded

stdout:
10.201.0.2 dev enp1s0f1np1 src 10.201.0.1 uid 1001 
    cache

$ ping -n -c 5 -W 1 -I enp1s0f1np1 10.201.0.2
return code: 0
status: succeeded

stdout:
PING 10.201.0.2 (10.201.0.2) from 10

,local_address,peer_address,expected_interface,route_succeeded,ping_succeeded
0,10.200.0.1,10.200.0.2,enP2p1s0f1np1,True,True
1,10.201.0.1,10.201.0.2,enp1s0f1np1,True,True


In [8]:
if which("ibv_devinfo") is None:
    print("ibv_devinfo is not installed.")
else:
    ibv_devinfo_result = run_command(("ibv_devinfo", "-v"), timeout=30.0)
    print_command_result(ibv_devinfo_result)

$ ibv_devinfo -v
return code: 0
status: succeeded

stdout:
hca_id:	rocep1s0f0
	transport:			InfiniBand (0)
	fw_ver:				28.45.4028
	node_guid:			4cbb:4703:0083:0241
	sys_image_guid:			4cbb:4703:0083:0241
	vendor_id:			0x02c9
	vendor_part_id:			4129
	hw_ver:				0x0
	board_id:			NVD0000000087
	phys_port_cnt:			1
	max_mr_size:			0xffffffffffffffff
	page_size_cap:			0xfffffffffffff000
	max_qp:				131072
	max_qp_wr:			8192
	device_cap_flags:		0x25321c36
					BAD_PKEY_CNTR
					BAD_QKEY_CNTR
					AUTO_PATH_MIG
					CHANGE_PHY_PORT
					PORT_ACTIVE_EVENT
					SYS_IMAGE_GUID
					RC_RNR_NAK_GEN
					MEM_WINDOW
					XRC
					MEM_MGT_EXTENSIONS
					MEM_WINDOW_TYPE_2B
					RAW_IP_CSUM
					MANAGED_FLOW_STEERING
	max_sge:			30
	max_sge_rd:			30
	max_cq:				16777216
	max_cqe:			4194303
	max_mr:				16777216
	max_pd:				8388608
	max_qp_rd_atom:			16
	max_ee_rd_atom:			0
	max_res_rd_atom:		2097152
	max_qp_init_rd_atom:		16
	max_ee_init_rd_atom:		0
	atomic_cap:			ATOMIC_HCA (1)
	max_ee:				0
	max_rdd:	

### Prediction: reliable-connected verbs communication

The `10.200.0.0/30` path is reachable over the active `roceP2p1s0f1` adapter, and GID index 3 represents its IPv4 RoCE v2 address.

I predict that:

1. the peer exposes a matching active RDMA adapter and IPv4 RoCE v2 GID;
2. a reliable-connected queue pair can transition to the ready states on both hosts;
3. `ibv_rc_pingpong` can exchange messages across the direct link;
4. the test will report successful iterations without packet loss or retry failure.

This is a functional smoke test. Its reported throughput and latency are not yet treated as controlled benchmark results.

In [9]:
peer_host = "spark-f868"

remote_inventory_commands = (
    ("hostname",),
    ("ip", "-brief", "address", "show", "dev", "enP2p1s0f1np1"),
    ("rdma", "link", "show", "roceP2p1s0f1/1"),
    ("ibv_devinfo", "-d", "roceP2p1s0f1"),
)

remote_inventory_results = []

for remote_command in remote_inventory_commands:
    result = run_command(
        (
            "ssh",
            "-o",
            "BatchMode=yes",
            "-o",
            "ConnectTimeout=5",
            peer_host,
            *remote_command,
        ),
        timeout=15.0,
    )
    remote_inventory_results.append(result)
    print_command_result(result)

if not all(result.succeeded for result in remote_inventory_results):
    raise RuntimeError("Remote inventory failed; do not start the verbs experiment.")

$ ssh -o BatchMode=yes -o ConnectTimeout=5 spark-f868 hostname
return code: 0
status: succeeded

stdout:
spark-f868

$ ssh -o BatchMode=yes -o ConnectTimeout=5 spark-f868 ip -brief address show dev enP2p1s0f1np1
return code: 0
status: succeeded

stdout:
enP2p1s0f1np1    UP             10.200.0.2/30

$ ssh -o BatchMode=yes -o ConnectTimeout=5 spark-f868 rdma link show roceP2p1s0f1/1
return code: 0
status: succeeded

stdout:
link roceP2p1s0f1/1 state ACTIVE physical_state LINK_UP netdev enP2p1s0f1np1

$ ssh -o BatchMode=yes -o ConnectTimeout=5 spark-f868 ibv_devinfo -d roceP2p1s0f1
return code: 0
status: succeeded

stdout:
hca_id:	roceP2p1s0f1
	transport:			InfiniBand (0)
	fw_ver:				28.45.4028
	node_guid:			4cbb:4703:0082:f86e
	sys_image_guid:			4cbb:4703:0082:f869
	vendor_id:			0x02c9
	vendor_part_id:			4129
	hw_ver:				0x0
	board_id:			NVD0000000087
	phys_port_cnt:			1
		port:	1
			state:			PORT_ACTIVE (4)
			max_mtu:		4096 (5)
			active_mtu:		4096 (5)
			sm_lid:			0
			port_lid:		0
	

### Reliable-connected ping-pong

`ibv_rc_pingpong` creates registered memory, completion queues, and a reliable-connected queue pair on each host. The server waits for a connection; the client connects through the peer's IPv4 address and exchanges a small number of messages.

The SSH process is retained as the server-process handle so that its stdout, stderr, exit status, and cleanup can all be recorded.

In [10]:
import subprocess
import time


rdma_device = "roceP2p1s0f1"
rdma_port = "1"
gid_index = "3"
peer_address = "10.200.0.2"

server_command = (
    "ssh",
    "-o",
    "BatchMode=yes",
    "-o",
    "ConnectTimeout=5",
    peer_host,
    "timeout",
    "20s",
    "ibv_rc_pingpong",
    "-d",
    rdma_device,
    "-i",
    rdma_port,
    "-g",
    gid_index,
)

client_command = (
    "timeout",
    "15s",
    "ibv_rc_pingpong",
    "-d",
    rdma_device,
    "-i",
    rdma_port,
    "-g",
    gid_index,
    peer_address,
)

print("$ " + " ".join(server_command))
server_process = subprocess.Popen(
    server_command,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
)

try:
    time.sleep(1.0)

    client_result = run_command(client_command, timeout=20.0)
    print_command_result(client_result)

    try:
        server_stdout, server_stderr = server_process.communicate(timeout=10.0)
    except subprocess.TimeoutExpired:
        server_process.terminate()
        try:
            server_stdout, server_stderr = server_process.communicate(timeout=5.0)
        except subprocess.TimeoutExpired:
            server_process.kill()
            server_stdout, server_stderr = server_process.communicate()

    server_returncode = server_process.returncode

finally:
    if server_process.poll() is None:
        server_process.terminate()
        try:
            server_process.wait(timeout=5.0)
        except subprocess.TimeoutExpired:
            server_process.kill()
            server_process.wait()

print("Remote server result")
print(f"return code: {server_returncode}")

if server_stdout.strip():
    print("\nstdout:")
    print(server_stdout.rstrip())

if server_stderr.strip():
    print("\nstderr:")
    print(server_stderr.rstrip())

rc_pingpong_succeeded = client_result.succeeded and server_returncode == 0

print(f"\nRC ping-pong succeeded: {rc_pingpong_succeeded}")

$ ssh -o BatchMode=yes -o ConnectTimeout=5 spark-f868 timeout 20s ibv_rc_pingpong -d roceP2p1s0f1 -i 1 -g 3
$ timeout 15s ibv_rc_pingpong -d roceP2p1s0f1 -i 1 -g 3 10.200.0.2
return code: 0
status: succeeded

stdout:
  local address:  LID 0x0000, QPN 0x0008ae, PSN 0x782e01, GID ::ffff:10.200.0.1
  remote address: LID 0x0000, QPN 0x0008ae, PSN 0x0a7a58, GID ::ffff:10.200.0.2
8192000 bytes in 0.01 seconds = 7864.63 Mbit/sec
1000 iters in 0.01 seconds = 8.33 usec/iter

Remote server result
return code: 0

stdout:
  local address:  LID 0x0000, QPN 0x0008ae, PSN 0x0a7a58, GID ::ffff:10.200.0.2
  remote address: LID 0x0000, QPN 0x0008ae, PSN 0x782e01, GID ::ffff:10.200.0.1
8192000 bytes in 0.01 seconds = 6845.92 Mbit/sec
1000 iters in 0.01 seconds = 9.57 usec/iter

RC ping-pong succeeded: True


In [11]:
rc_pingpong_summary_df = pd.DataFrame(
    [
        {
            "local_host": socket.gethostname(),
            "peer_host": peer_host,
            "rdma_device": rdma_device,
            "port": int(rdma_port),
            "gid_index": int(gid_index),
            "peer_address": peer_address,
            "client_returncode": client_result.returncode,
            "server_returncode": server_returncode,
            "succeeded": rc_pingpong_succeeded,
        }
    ]
)

display(rc_pingpong_summary_df)

,local_host,peer_host,rdma_device,port,gid_index,peer_address,client_returncode,server_returncode,succeeded
0,spark-0240,spark-f868,roceP2p1s0f1,1,3,10.200.0.2,0,0,True


### Prediction: RDMA write latency across both direct links

Both active ports report the same nominal 200 Gb/s rate and use equivalent
`mlx5_core` devices. For a small 8-byte RDMA write, serialization time is
negligible; queue-pair processing, PCIe traversal, NIC processing, and software
measurement overhead should dominate.

I predict:

1. both direct links will complete all trials successfully;
2. median latency will be measured in microseconds rather than the roughly
   millisecond-scale round-trip time seen with ICMP;
3. the two links will have similar latency distributions;
4. trial-to-trial variation may be larger than the systematic difference
   between the links.

CPU 15 is used on both hosts to reduce process migration. This does not isolate
the CPU or eliminate operating-system noise.

In [12]:
latency_paths = (
    {
        "path": "10.200",
        "local_address": "10.200.0.1",
        "peer_address": "10.200.0.2",
        "rdma_device": "roceP2p1s0f1",
    },
    {
        "path": "10.201",
        "local_address": "10.201.0.1",
        "peer_address": "10.201.0.2",
        "rdma_device": "rocep1s0f1",
    },
)

LATENCY_CPU = 15
LATENCY_PORT = 1
LATENCY_GID_INDEX = 3
LATENCY_MESSAGE_BYTES = 8
LATENCY_ITERATIONS = 10_000
LATENCY_TRIALS = 5

latency_paths

({'path': '10.200',
  'local_address': '10.200.0.1',
  'peer_address': '10.200.0.2',
  'rdma_device': 'roceP2p1s0f1'},
 {'path': '10.201',
  'local_address': '10.201.0.1',
  'peer_address': '10.201.0.2',
  'rdma_device': 'rocep1s0f1'})

In [ ]:
def parse_perftest_latency(output: str) -> dict[str, float | int] | None:
    data_rows = []

    for line in output.splitlines():
        fields = line.split()

        if len(fields) not in {7, 9}:
            continue

        try:
            size = int(fields[0])
            iterations = int(fields[1])
            minimum = float(fields[2])
            maximum = float(fields[3])
            typical = float(fields[4])
            average = float(fields[5])
            standard_deviation = float(fields[6])

            percentile_99 = float(fields[7]) if len(fields) >= 8 else None
            percentile_99_9 = float(fields[8]) if len(fields) >= 9 else None
        except ValueError:
            continue

        data_rows.append(
            {
                "message_bytes": size,
                "iterations": iterations,
                "minimum_us": minimum,
                "maximum_us": maximum,
                "typical_us": typical,
                "average_us": average,
                "standard_deviation_us": standard_deviation,
                "percentile_99_us": percentile_99,
                "percentile_99_9_us": percentile_99_9,
            }
        )

    return data_rows[-1] if data_rows else None

In [42]:
def run_write_latency_trial(
    *,
    trial: int,
    path: dict[str, str],
    control_port: int,
    cpu: int = LATENCY_CPU,
) -> dict[str, object]:
    device = path["rdma_device"]
    peer_address = path["peer_address"]

    common_arguments = (
        "taskset",
        "-c",
        str(cpu),
        "ib_write_lat",
        "-d",
        device,
        "-i",
        str(LATENCY_PORT),
        "-x",
        str(LATENCY_GID_INDEX),
        "-s",
        str(LATENCY_MESSAGE_BYTES),
        "-n",
        str(LATENCY_ITERATIONS),
        "-p",
        str(control_port),
        "-F",
    )

    server_command = (
        "ssh",
        "-T",
        "-o",
        "BatchMode=yes",
        "-o",
        "ConnectTimeout=5",
        peer_host,
        "timeout",
        "30s",
        *common_arguments,
    )

    client_command = (
        "timeout",
        "25s",
        *common_arguments,
        peer_address,
    )

    server_process = subprocess.Popen(
        server_command,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
    )

    client_result = None
    server_stdout = ""
    server_stderr = ""

    try:
        time.sleep(1.0)
        client_result = run_command(client_command, timeout=30.0)

        try:
            server_stdout, server_stderr = server_process.communicate(timeout=10.0)
        except subprocess.TimeoutExpired:
            server_process.terminate()

            try:
                server_stdout, server_stderr = server_process.communicate(timeout=5.0)
            except subprocess.TimeoutExpired:
                server_process.kill()
                server_stdout, server_stderr = server_process.communicate()
    finally:
        if server_process.poll() is None:
            server_process.terminate()

            try:
                server_process.wait(timeout=5.0)
            except subprocess.TimeoutExpired:
                server_process.kill()
                server_process.wait()

    server_returncode = server_process.returncode
    parsed = (
        parse_perftest_latency(client_result.stdout)
        if client_result is not None
        else None
    )

    transport_succeeded = (
        client_result is not None and client_result.succeeded and server_returncode == 0
    )
    parse_succeeded = parsed is not None
    succeeded = transport_succeeded and parse_succeeded

    return {
        "trial": trial,
        "cpu": cpu,
        **path,
        "control_port": control_port,
        "client_returncode": (
            client_result.returncode if client_result is not None else None
        ),
        "server_returncode": server_returncode,
        "transport_succeeded": transport_succeeded,
        "parse_succeeded": parse_succeeded,
        "succeeded": succeeded,
        "client_stdout": (client_result.stdout if client_result is not None else ""),
        "client_stderr": (client_result.stderr if client_result is not None else ""),
        "server_stdout": server_stdout,
        "server_stderr": server_stderr,
        **(parsed or {}),
    }

In [29]:
latency_measurements = []

for trial in range(1, LATENCY_TRIALS + 1):
    ordered_paths = latency_paths if trial % 2 == 1 else tuple(reversed(latency_paths))

    for path_index, path in enumerate(ordered_paths):
        control_port = 18_600 + trial * 10 + path_index

        print(f"Trial {trial}: path {path['path']} through {path['rdma_device']}")

        measurement = run_write_latency_trial(
            trial=trial,
            path=path,
            control_port=control_port,
        )
        latency_measurements.append(measurement)

        print(
            f"  transport_succeeded={measurement['transport_succeeded']}, "
            f"parse_succeeded={measurement['parse_succeeded']}, "
            f"typical_us={measurement.get('typical_us')}, "
            f"average_us={measurement.get('average_us')}"
        )

        if not measurement["transport_succeeded"]:
            print("\nClient stdout:")
            print(measurement["client_stdout"])
            print("\nClient stderr:")
            print(measurement["client_stderr"])
            print("\nServer stdout:")
            print(measurement["server_stdout"])
            print("\nServer stderr:")
            print(measurement["server_stderr"])
            raise RuntimeError(f"Latency trial failed on path {path['path']}")

        if not measurement["parse_succeeded"]:
            print("\nBenchmark completed, but its output could not be parsed.")
            print("\nClient stdout:")
            print(measurement["client_stdout"])
            raise RuntimeError(f"Could not parse latency result on path {path['path']}")

latency_results_df = pd.DataFrame(latency_measurements)

Trial 1: path 10.200 through roceP2p1s0f1
  transport_succeeded=True, parse_succeeded=True, typical_us=1.7, average_us=1.81
Trial 1: path 10.201 through rocep1s0f1
  transport_succeeded=True, parse_succeeded=True, typical_us=1.42, average_us=1.43
Trial 2: path 10.201 through rocep1s0f1
  transport_succeeded=True, parse_succeeded=True, typical_us=1.43, average_us=1.43
Trial 2: path 10.200 through roceP2p1s0f1
  transport_succeeded=True, parse_succeeded=True, typical_us=1.7, average_us=1.82
Trial 3: path 10.200 through roceP2p1s0f1
  transport_succeeded=True, parse_succeeded=True, typical_us=1.7, average_us=1.81
Trial 3: path 10.201 through rocep1s0f1
  transport_succeeded=True, parse_succeeded=True, typical_us=1.43, average_us=1.43
Trial 4: path 10.201 through rocep1s0f1
  transport_succeeded=True, parse_succeeded=True, typical_us=1.43, average_us=1.43
Trial 4: path 10.200 through roceP2p1s0f1
  transport_succeeded=True, parse_succeeded=True, typical_us=1.71, average_us=1.82
Trial 5: pa

In [31]:
latency_columns = [
    "trial",
    "path",
    "rdma_device",
    "message_bytes",
    "iterations",
    "minimum_us",
    "maximum_us",
    "typical_us",
    "average_us",
    "standard_deviation_us",
    "percentile_99_us",
    "percentile_99_9_us",
    "succeeded",
]

display(latency_results_df[latency_columns])

,trial,path,rdma_device,message_bytes,iterations,minimum_us,maximum_us,typical_us,average_us,standard_deviation_us,percentile_99_us,percentile_99_9_us,succeeded
0,1,10.200,roceP2p1s0f1,8,10000,1.64,6.06,1.70,1.81,0.02,2.21,2.31,True
1,1,10.201,rocep1s0f1,8,10000,1.38,3.34,1.42,1.43,0.01,1.53,1.82,True
2,2,10.201,rocep1s0f1,8,10000,1.36,2.41,1.43,1.43,0.00,1.53,1.88,True
3,2,10.200,roceP2p1s0f1,8,10000,1.65,2.60,1.70,1.82,0.00,2.21,2.30,True
4,3,10.200,roceP2p1s0f1,8,10000,1.66,3.10,1.70,1.81,0.00,2.21,2.30,True
5,3,10.201,rocep1s0f1,8,10000,1.38,3.01,1.43,1.43,0.00,1.53,1.82,True
6,4,10.201,rocep1s0f1,8,10000,1.38,20.56,1.43,1.43,0.07,1.53,1.82,True
7,4,10.200,roceP2p1s0f1,8,10000,1.66,2.89,1.71,1.82,0.01,2.22,2.30,True
8,5,10.200,roceP2p1s0f1,8,10000,1.63,3.07,1.71,1.82,0.00,2.22,2.30,True
9,5,10.201,rocep1s0f1,8,10000,1.37,2.36,1.43,1.43,0.00,1.53,1.79,True


In [32]:
latency_summary_df = latency_results_df.groupby("path", as_index=False).agg(
    trials=("trial", "count"),
    minimum_us=("minimum_us", "min"),
    median_typical_us=("typical_us", "median"),
    mean_typical_us=("typical_us", "mean"),
    minimum_average_us=("average_us", "min"),
    median_average_us=("average_us", "median"),
    mean_average_us=("average_us", "mean"),
    maximum_average_us=("average_us", "max"),
)

display(latency_summary_df)

,path,trials,minimum_us,median_typical_us,mean_typical_us,minimum_average_us,median_average_us,mean_average_us,maximum_average_us
0,10.200,5,1.63,1.70,1.704,1.81,1.82,1.816,1.82
1,10.201,5,1.36,1.43,1.428,1.43,1.43,1.430,1.43


In [33]:
latency_pairs_df = latency_results_df.pivot(
    index="trial",
    columns="path",
    values="average_us",
).reset_index()

latency_pairs_df["10.201_minus_10.200_us"] = (
    latency_pairs_df["10.201"] - latency_pairs_df["10.200"]
)
latency_pairs_df["10.201_over_10.200"] = (
    latency_pairs_df["10.201"] / latency_pairs_df["10.200"]
)

display(latency_pairs_df)

print(
    "Median paired difference "
    f"(10.201 - 10.200): "
    f"{latency_pairs_df['10.201_minus_10.200_us'].median():.3f} µs"
)

path,trial,10.200,10.201,10.201_minus_10.200_us,10.201_over_10.200
0,1,1.81,1.43,-0.38,0.790055
1,2,1.82,1.43,-0.39,0.785714
2,3,1.81,1.43,-0.38,0.790055
3,4,1.82,1.43,-0.39,0.785714
4,5,1.82,1.43,-0.39,0.785714


Median paired difference (10.201 - 10.200): -0.390 µs


In [34]:
first_latency_output = latency_results_df.iloc[0]

print("Client output from the first successful trial:")
print(first_latency_output["client_stdout"])

print("\nServer output from the first successful trial:")
print(first_latency_output["server_stdout"])

Client output from the first successful trial:
---------------------------------------------------------------------------------------
                    RDMA_Write Latency Test
 Dual-port       : OFF		Device         : roceP2p1s0f1
 Number of qps   : 1		Transport type : IB
 Connection type : RC		Using SRQ      : OFF
 PCIe relax order: OFF
 ibv_wr* API     : ON
 TX depth        : 1
 Mtu             : 4096[B]
 Link type       : Ethernet
 GID index       : 3
 Max inline data : 220[B]
 rdma_cm QPs	 : OFF
 Data ex. method : Ethernet
---------------------------------------------------------------------------------------
 local address: LID 0000 QPN 0x08c0 PSN 0x195852 RKey 0x20e3f3 VAddr 0x00afdb54238000
 GID: 00:00:00:00:00:00:00:00:00:00:255:255:10:200:00:01
 remote address: LID 0000 QPN 0x08c0 PSN 0x3b43b9 RKey 0x21fc00 VAddr 0x00bc738cac0000
 GID: 00:00:00:00:00:00:00:00:00:00:255:255:10:200:00:02
---------------------------------------------------------------------------------------
 #

### Prediction: CPU and PCIe locality

The two active RDMA devices occupy different PCI domains:

- `rocep1s0f1` is backed by `0000:01:00.1`;
- `roceP2p1s0f1` is backed by `0002:01:00.1`.

The stable latency difference may reflect a different path between CPU 15,
system memory, and each NIC.

I predict that Linux topology metadata will expose either different NUMA-node,
CPU-affinity, or PCIe ancestry information for the two devices. If the platform
reports only one NUMA node, PCIe ancestry may still differ even though the NUMA
identifier is the same.

In [35]:
active_rdma_topology = (
    {
        "path": "10.200",
        "rdma_device": "roceP2p1s0f1",
        "pci_device": "0002:01:00.1",
        "network_interface": "enP2p1s0f1np1",
    },
    {
        "path": "10.201",
        "rdma_device": "rocep1s0f1",
        "pci_device": "0000:01:00.1",
        "network_interface": "enp1s0f1np1",
    },
)

topology_rows = []

for device in active_rdma_topology:
    pci_path = Path("/sys/bus/pci/devices") / device["pci_device"]
    net_path = Path("/sys/class/net") / device["network_interface"]

    topology_rows.append(
        {
            **device,
            "numa_node": read_text(pci_path / "numa_node"),
            "local_cpulist": read_text(pci_path / "local_cpulist"),
            "local_cpus": read_text(pci_path / "local_cpus"),
            "irq_affinity_hint": read_text(net_path / "device" / "local_cpulist"),
            "pci_path": str(pci_path.resolve()),
        }
    )

rdma_topology_df = pd.DataFrame(topology_rows)
display(rdma_topology_df)

,path,rdma_device,pci_device,network_interface,numa_node,local_cpulist,local_cpus,irq_affinity_hint,pci_path
0,10.200,roceP2p1s0f1,0002:01:00.1,enP2p1s0f1np1,-1,0-19,fffff,0-19,/sys/devices/pci0002:00/0002:00:00.0/0002:01:00.1
1,10.201,rocep1s0f1,0000:01:00.1,enp1s0f1np1,-1,0-19,fffff,0-19,/sys/devices/pci0000:00/0000:00:00.0/0000:01:00.1


In [36]:
for device in active_rdma_topology:
    print(f"Path {device['path']}: {device['pci_device']}")

    result = run_command(
        (
            "lspci",
            "-D",
            "-s",
            device["pci_device"],
            "-vv",
        ),
        timeout=15.0,
    )
    print_command_result(result)

Path 10.200: 0002:01:00.1
$ lspci -D -s 0002:01:00.1 -vv
return code: 0
status: succeeded

stdout:
0002:01:00.1 Ethernet controller: Mellanox Technologies MT2910 Family [ConnectX-7]
	Subsystem: Mellanox Technologies MT2910 Family [ConnectX-7]
	Control: I/O- Mem+ BusMaster+ SpecCycle- MemWINV- VGASnoop- ParErr- Stepping- SERR- FastB2B- DisINTx+
	Status: Cap+ 66MHz- UDF- FastB2B- ParErr- DEVSEL=fast >TAbort- <TAbort- <MAbort- >SERR- <PERR- INTx-
	Latency: 0
	Interrupt: pin B routed to IRQ 405
	IOMMU group: 11
	Region 0: Memory at 3d02000000 (64-bit, prefetchable) [size=32M]
	Expansion ROM at 5d200000 [virtual] [disabled] [size=1M]
	Capabilities: <access denied>
	Kernel driver in use: mlx5_core
	Kernel modules: mlx5_core

Path 10.201: 0000:01:00.1
$ lspci -D -s 0000:01:00.1 -vv
return code: 0
status: succeeded

stdout:
0000:01:00.1 Ethernet controller: Mellanox Technologies MT2910 Family [ConnectX-7]
	Subsystem: Mellanox Technologies MT2910 Family [ConnectX-7]
	Control: I/O- Mem+ BusMaste

In [37]:
cpu_topology_rows = []

for cpu in range(os.cpu_count() or 0):
    cpu_path = Path(f"/sys/devices/system/cpu/cpu{cpu}/topology")

    cpu_topology_rows.append(
        {
            "cpu": cpu,
            "core_id": read_text(cpu_path / "core_id"),
            "cluster_id": read_text(cpu_path / "cluster_id"),
            "package_id": read_text(cpu_path / "physical_package_id"),
            "core_type": read_text(
                Path(f"/sys/devices/system/cpu/cpu{cpu}/cpufreq/cpuinfo_max_freq")
            ),
        }
    )

cpu_topology_df = pd.DataFrame(cpu_topology_rows)
display(cpu_topology_df)

,cpu,core_id,cluster_id,package_id,core_type
0,0,0,56,36,2808000
1,1,1,56,36,2808000
2,2,2,56,36,2808000
3,3,3,56,36,2808000
4,4,4,56,36,2808000
5,5,5,56,36,3900000
6,6,6,56,36,3900000
7,7,7,56,36,3900000
8,8,8,56,36,3900000
9,9,9,56,36,3900000


In [38]:
for device in active_rdma_topology:
    interface = device["network_interface"]

    print(f"IRQ entries associated with {interface}")

    irq_result = run_command(
        (
            "bash",
            "-lc",
            f"grep -i -- {interface!r} /proc/interrupts || true",
        ),
        timeout=10.0,
    )
    print_command_result(irq_result)

IRQ entries associated with enP2p1s0f1np1
$ bash -lc grep -i -- 'enP2p1s0f1np1' /proc/interrupts || true
return code: 0
status: succeeded

IRQ entries associated with enp1s0f1np1
$ bash -lc grep -i -- 'enp1s0f1np1' /proc/interrupts || true
return code: 0
status: succeeded



In [39]:
interrupt_lines = Path("/proc/interrupts").read_text().splitlines()

for device in active_rdma_topology:
    interface = device["network_interface"]

    matching_lines = [
        line for line in interrupt_lines if interface.lower() in line.lower()
    ]

    print(f"IRQ entries associated with {interface}")

    if matching_lines:
        print("\n".join(matching_lines))
    else:
        print("No interface-name matches found in /proc/interrupts.")

    print()

IRQ entries associated with enP2p1s0f1np1
No interface-name matches found in /proc/interrupts.

IRQ entries associated with enp1s0f1np1
No interface-name matches found in /proc/interrupts.



### Prediction: CPU placement and RDMA write latency

The previous measurements used CPU 15 on both hosts and showed a stable latency
advantage for the `10.201` path.

The two NICs occupy separate PCI root domains, but Linux exposes neither device
as NUMA-local to a subset of CPUs. A CPU-placement sweep can reveal locality
that is not represented by the generic NUMA metadata.

I predict:

1. Cortex-X925 cores will produce lower or more stable measured latency than
   Cortex-A725 cores because the benchmark's posting and completion loop runs
   on the CPU.
2. If one CPU cluster is closer to one PCI root domain, the relative advantage
   of the two network paths may change between clusters.
3. If `10.201` remains faster by approximately the same amount for every CPU,
   the difference is more likely associated with the NIC paths themselves than
   with the selected initiating core.

The experiment still measures the combined configuration of both hosts because
the client and server are pinned to corresponding CPU numbers.

In [40]:
cpu_placement_cases = (
    {
        "cpu": 0,
        "cluster_id": 56,
        "core_type": "Cortex-A725",
    },
    {
        "cpu": 5,
        "cluster_id": 56,
        "core_type": "Cortex-X925",
    },
    {
        "cpu": 10,
        "cluster_id": 1144,
        "core_type": "Cortex-A725",
    },
    {
        "cpu": 15,
        "cluster_id": 1144,
        "core_type": "Cortex-X925",
    },
)

CPU_PLACEMENT_TRIALS = 3

In [ ]:
cpu_latency_measurements = []

measurement_index = 0

for trial in range(1, CPU_PLACEMENT_TRIALS + 1):
    ordered_cpu_cases = (
        cpu_placement_cases if trial % 2 == 1 else tuple(reversed(cpu_placement_cases))
    )

    for cpu_case_index, cpu_case in enumerate(ordered_cpu_cases):
        ordered_paths = (
            latency_paths
            if (trial + cpu_case_index) % 2 == 0
            else tuple(reversed(latency_paths))
        )

        for path in ordered_paths:
            measurement_index += 1
            control_port = 18_800 + measurement_index

            print(
                f"Trial {trial}, CPU {cpu_case['cpu']} "
                f"({cpu_case['core_type']}, cluster "
                f"{cpu_case['cluster_id']}), path {path['path']}"
            )

            measurement = run_write_latency_trial(
                trial=trial,
                path=path,
                control_port=control_port,
                cpu=cpu_case["cpu"],
            )

            measurement.update(
                {
                    "cluster_id": cpu_case["cluster_id"],
                    "core_type": cpu_case["core_type"],
                }
            )
            cpu_latency_measurements.append(measurement)

            print(
                f"  transport={measurement['transport_succeeded']}, "
                f"parse={measurement['parse_succeeded']}, "
                f"average_us={measurement.get('average_us')}"
            )

            if not measurement["transport_succeeded"]:
                print("\nClient stdout:")
                print(measurement["client_stdout"])
                print("\nClient stderr:")
                print(measurement["client_stderr"])
                print("\nServer stdout:")
                print(measurement["server_stdout"])
                print("\nServer stderr:")
                print(measurement["server_stderr"])
                raise RuntimeError(
                    f"Transport failed on CPU {cpu_case['cpu']}, path {path['path']}"
                )

            if not measurement["parse_succeeded"]:
                print("\nClient stdout:")
                print(measurement["client_stdout"])
                raise RuntimeError(
                    f"Parsing failed on CPU {cpu_case['cpu']}, path {path['path']}"
                )

cpu_latency_results_df = pd.DataFrame(cpu_latency_measurements)

Trial 1, CPU 0 (Cortex-A725, cluster 56), path 10.201
  transport=True, parse=True, average_us=1.73
Trial 1, CPU 0 (Cortex-A725, cluster 56), path 10.200
  transport=True, parse=True, average_us=1.98
Trial 1, CPU 5 (Cortex-X925, cluster 56), path 10.200
  transport=True, parse=True, average_us=1.85
Trial 1, CPU 5 (Cortex-X925, cluster 56), path 10.201
  transport=True, parse=True, average_us=1.46
Trial 1, CPU 10 (Cortex-A725, cluster 1144), path 10.201
  transport=True, parse=True, average_us=1.74
Trial 1, CPU 10 (Cortex-A725, cluster 1144), path 10.200
  transport=True, parse=True, average_us=2.0
Trial 1, CPU 15 (Cortex-X925, cluster 1144), path 10.200
  transport=True, parse=True, average_us=1.82
Trial 1, CPU 15 (Cortex-X925, cluster 1144), path 10.201
  transport=True, parse=True, average_us=1.43
Trial 2, CPU 15 (Cortex-X925, cluster 1144), path 10.200
  transport=True, parse=True, average_us=1.82
Trial 2, CPU 15 (Cortex-X925, cluster 1144), path 10.201
  transport=True, parse=True,

In [44]:
display(
    cpu_latency_results_df[
        [
            "trial",
            "cpu",
            "cluster_id",
            "core_type",
            "path",
            "typical_us",
            "average_us",
            "percentile_99_us",
            "percentile_99_9_us",
            "succeeded",
        ]
    ]
)

,trial,cpu,cluster_id,core_type,path,typical_us,average_us,percentile_99_us,percentile_99_9_us,succeeded
0,1,0,56,Cortex-A725,10.201,1.76,1.73,2.07,2.28,True
1,1,0,56,Cortex-A725,10.200,2.02,1.98,2.31,2.74,True
2,1,5,56,Cortex-X925,10.200,1.76,1.85,2.26,2.34,True
3,1,5,56,Cortex-X925,10.201,1.46,1.46,1.56,2.19,True
4,1,10,1144,Cortex-A725,10.201,1.77,1.74,2.07,2.41,True
5,1,10,1144,Cortex-A725,10.200,2.04,2.00,2.34,2.74,True
6,1,15,1144,Cortex-X925,10.200,1.71,1.82,2.22,2.30,True
7,1,15,1144,Cortex-X925,10.201,1.43,1.43,1.48,1.76,True
8,2,15,1144,Cortex-X925,10.200,1.71,1.82,2.21,2.30,True
9,2,15,1144,Cortex-X925,10.201,1.43,1.44,1.54,1.83,True


In [ ]:
cpu_latency_summary_df = cpu_latency_results_df.groupby(
    ["cpu", "cluster_id", "core_type", "path"],
    as_index=False,
).agg(
    trials=("trial", "count"),
    median_typical_us=("typical_us", "median"),
    median_average_us=("average_us", "median"),
    mean_average_us=("average_us", "mean"),
    median_99_us=("percentile_99_us", "median"),
    median_99_9_us=("percentile_99_9_us", "median"),
)

display(cpu_latency_summary_df)

,cpu,cluster_id,core_type,path,trials,median_typical_us,median_average_us,mean_average_us,median_99_us,median_99_9_us
0,0,56,Cortex-A725,10.200,3,2.02,1.98,1.983333,2.31,2.62
1,0,56,Cortex-A725,10.201,3,1.77,1.74,1.736667,2.07,2.33
2,5,56,Cortex-X925,10.200,3,1.75,1.85,1.846667,2.26,2.34
3,5,56,Cortex-X925,10.201,3,1.46,1.47,1.466667,1.56,1.84
4,10,1144,Cortex-A725,10.200,3,2.04,1.99,1.993333,2.34,2.74
5,10,1144,Cortex-A725,10.201,3,1.77,1.74,1.743333,2.08,2.41
6,15,1144,Cortex-X925,10.200,3,1.71,1.82,1.820000,2.22,2.30
7,15,1144,Cortex-X925,10.201,3,1.43,1.43,1.433333,1.52,1.83


In [46]:
cpu_path_pairs_df = cpu_latency_results_df.pivot(
    index=["trial", "cpu", "cluster_id", "core_type"],
    columns="path",
    values="average_us",
).reset_index()

cpu_path_pairs_df["10.201_minus_10.200_us"] = (
    cpu_path_pairs_df["10.201"] - cpu_path_pairs_df["10.200"]
)
cpu_path_pairs_df["10.201_over_10.200"] = (
    cpu_path_pairs_df["10.201"] / cpu_path_pairs_df["10.200"]
)

display(cpu_path_pairs_df)

path,trial,cpu,cluster_id,core_type,10.200,10.201,10.201_minus_10.200_us,10.201_over_10.200
0,1,0,56,Cortex-A725,1.98,1.73,-0.25,0.873737
1,1,5,56,Cortex-X925,1.85,1.46,-0.39,0.789189
2,1,10,1144,Cortex-A725,2.00,1.74,-0.26,0.870000
3,1,15,1144,Cortex-X925,1.82,1.43,-0.39,0.785714
4,2,0,56,Cortex-A725,1.98,1.74,-0.24,0.878788
5,2,5,56,Cortex-X925,1.84,1.47,-0.37,0.798913
6,2,10,1144,Cortex-A725,1.99,1.74,-0.25,0.874372
7,2,15,1144,Cortex-X925,1.82,1.44,-0.38,0.791209
8,3,0,56,Cortex-A725,1.99,1.74,-0.25,0.874372
9,3,5,56,Cortex-X925,1.85,1.47,-0.38,0.794595


In [ ]:
cpu_path_difference_summary_df = cpu_path_pairs_df.groupby(
    ["cpu", "cluster_id", "core_type"],
    as_index=False,
).agg(
    median_10_200_us=("10.200", "median"),
    median_10_201_us=("10.201", "median"),
    median_difference_us=("10.201_minus_10.200_us", "median"),
    median_ratio=("10.201_over_10.200", "median"),
)

display(cpu_path_difference_summary_df)

,cpu,cluster_id,core_type,median_10_200_us,median_10_201_us,median_difference_us,median_ratio
0,0,56,Cortex-A725,1.98,1.74,-0.25,0.874372
1,5,56,Cortex-X925,1.85,1.47,-0.38,0.794595
2,10,1144,Cortex-A725,1.99,1.74,-0.25,0.874372
3,15,1144,Cortex-X925,1.82,1.43,-0.39,0.785714


### Prediction: initiating versus target CPU

The previous experiment pinned the client and server to the same CPU type, so
the source of the core-type effect remained ambiguous.

An RDMA write is one-sided: after queue-pair setup and memory registration, the
initiating process posts writes while the remote NIC places the payload into
registered memory without requiring the target CPU to process each operation.

I therefore predict:

1. changing the client from a Cortex-A725 to a Cortex-X925 will materially
   reduce measured latency;
2. changing only the server CPU will have a smaller effect;
3. the `10.201` path will remain faster for every client/server CPU pairing;
4. if server CPU type substantially changes the result, then synchronization,
   completion handling, or benchmark protocol activity contributes more to the
   timed measurement than the simple one-sided model suggests.

In [ ]:
remote_cpu_cases = (0, 5)

remote_cpu_topology_rows = []

for cpu in remote_cpu_cases:
    topology_base = f"/sys/devices/system/cpu/cpu{cpu}/topology"
    frequency_path = f"/sys/devices/system/cpu/cpu{cpu}/cpufreq/cpuinfo_max_freq"

    for field, path in (
        ("cluster_id", f"{topology_base}/cluster_id"),
        ("core_id", f"{topology_base}/core_id"),
        ("maximum_frequency_khz", frequency_path),
    ):
        result = run_command(
            (
                "ssh",
                "-T",
                "-o",
                "BatchMode=yes",
                peer_host,
                "cat",
                path,
            ),
            timeout=10.0,
        )

        if not result.succeeded:
            raise RuntimeError(f"Could not read remote CPU {cpu} field {field}")

        remote_cpu_topology_rows.append(
            {
                "cpu": cpu,
                "field": field,
                "value": result.stdout.strip(),
            }
        )

remote_cpu_topology_df = pd.DataFrame(remote_cpu_topology_rows)
display(remote_cpu_topology_df)

,cpu,field,value
0,0,cluster_id,56
1,0,core_id,0
2,0,maximum_frequency_khz,2808000
3,5,cluster_id,56
4,5,core_id,5
5,5,maximum_frequency_khz,3900000


In [49]:
asymmetric_cpu_cases = (
    {
        "client_cpu": 0,
        "client_core_type": "Cortex-A725",
        "server_cpu": 0,
        "server_core_type": "Cortex-A725",
    },
    {
        "client_cpu": 0,
        "client_core_type": "Cortex-A725",
        "server_cpu": 5,
        "server_core_type": "Cortex-X925",
    },
    {
        "client_cpu": 5,
        "client_core_type": "Cortex-X925",
        "server_cpu": 0,
        "server_core_type": "Cortex-A725",
    },
    {
        "client_cpu": 5,
        "client_core_type": "Cortex-X925",
        "server_cpu": 5,
        "server_core_type": "Cortex-X925",
    },
)

ASYMMETRIC_CPU_TRIALS = 3

In [ ]:
def run_asymmetric_write_latency_trial(
    *,
    trial: int,
    path: dict[str, str],
    control_port: int,
    client_cpu: int,
    server_cpu: int,
) -> dict[str, object]:
    device = path["rdma_device"]
    peer_address = path["peer_address"]

    benchmark_arguments = (
        "ib_write_lat",
        "-d",
        device,
        "-i",
        str(LATENCY_PORT),
        "-x",
        str(LATENCY_GID_INDEX),
        "-s",
        str(LATENCY_MESSAGE_BYTES),
        "-n",
        str(LATENCY_ITERATIONS),
        "-p",
        str(control_port),
        "-F",
    )

    server_command = (
        "ssh",
        "-T",
        "-o",
        "BatchMode=yes",
        "-o",
        "ConnectTimeout=5",
        peer_host,
        "timeout",
        "30s",
        "taskset",
        "-c",
        str(server_cpu),
        *benchmark_arguments,
    )

    client_command = (
        "timeout",
        "25s",
        "taskset",
        "-c",
        str(client_cpu),
        *benchmark_arguments,
        peer_address,
    )

    server_process = subprocess.Popen(
        server_command,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
    )

    client_result = None
    server_stdout = ""
    server_stderr = ""

    try:
        time.sleep(1.0)
        client_result = run_command(client_command, timeout=30.0)

        try:
            server_stdout, server_stderr = server_process.communicate(timeout=10.0)
        except subprocess.TimeoutExpired:
            server_process.terminate()

            try:
                server_stdout, server_stderr = server_process.communicate(timeout=5.0)
            except subprocess.TimeoutExpired:
                server_process.kill()
                server_stdout, server_stderr = server_process.communicate()
    finally:
        if server_process.poll() is None:
            server_process.terminate()

            try:
                server_process.wait(timeout=5.0)
            except subprocess.TimeoutExpired:
                server_process.kill()
                server_process.wait()

    server_returncode = server_process.returncode
    parsed = (
        parse_perftest_latency(client_result.stdout)
        if client_result is not None
        else None
    )

    transport_succeeded = (
        client_result is not None and client_result.succeeded and server_returncode == 0
    )
    parse_succeeded = parsed is not None

    return {
        "trial": trial,
        "client_cpu": client_cpu,
        "server_cpu": server_cpu,
        **path,
        "control_port": control_port,
        "client_returncode": (
            client_result.returncode if client_result is not None else None
        ),
        "server_returncode": server_returncode,
        "transport_succeeded": transport_succeeded,
        "parse_succeeded": parse_succeeded,
        "succeeded": transport_succeeded and parse_succeeded,
        "client_stdout": (client_result.stdout if client_result is not None else ""),
        "client_stderr": (client_result.stderr if client_result is not None else ""),
        "server_stdout": server_stdout,
        "server_stderr": server_stderr,
        **(parsed or {}),
    }

In [ ]:
asymmetric_latency_measurements = []
measurement_index = 0

for trial in range(1, ASYMMETRIC_CPU_TRIALS + 1):
    ordered_cases = (
        asymmetric_cpu_cases
        if trial % 2 == 1
        else tuple(reversed(asymmetric_cpu_cases))
    )

    for case_index, cpu_case in enumerate(ordered_cases):
        ordered_paths = (
            latency_paths
            if (trial + case_index) % 2 == 0
            else tuple(reversed(latency_paths))
        )

        for path in ordered_paths:
            measurement_index += 1
            control_port = 19_000 + measurement_index

            print(
                f"Trial {trial}, client CPU {cpu_case['client_cpu']} "
                f"({cpu_case['client_core_type']}), server CPU "
                f"{cpu_case['server_cpu']} "
                f"({cpu_case['server_core_type']}), path {path['path']}"
            )

            measurement = run_asymmetric_write_latency_trial(
                trial=trial,
                path=path,
                control_port=control_port,
                client_cpu=cpu_case["client_cpu"],
                server_cpu=cpu_case["server_cpu"],
            )

            measurement.update(cpu_case)
            asymmetric_latency_measurements.append(measurement)

            print(
                f"  transport={measurement['transport_succeeded']}, "
                f"parse={measurement['parse_succeeded']}, "
                f"average_us={measurement.get('average_us')}"
            )

            if not measurement["succeeded"]:
                print("\nClient stdout:")
                print(measurement["client_stdout"])
                print("\nClient stderr:")
                print(measurement["client_stderr"])
                print("\nServer stdout:")
                print(measurement["server_stdout"])
                print("\nServer stderr:")
                print(measurement["server_stderr"])
                raise RuntimeError("Asymmetric CPU latency measurement failed")

asymmetric_latency_results_df = pd.DataFrame(asymmetric_latency_measurements)

Trial 1, client CPU 0 (Cortex-A725), server CPU 0 (Cortex-A725), path 10.201
  transport=True, parse=True, average_us=1.74
Trial 1, client CPU 0 (Cortex-A725), server CPU 0 (Cortex-A725), path 10.200
  transport=True, parse=True, average_us=1.99
Trial 1, client CPU 0 (Cortex-A725), server CPU 5 (Cortex-X925), path 10.200
  transport=True, parse=True, average_us=1.97
Trial 1, client CPU 0 (Cortex-A725), server CPU 5 (Cortex-X925), path 10.201
  transport=True, parse=True, average_us=1.61
Trial 1, client CPU 5 (Cortex-X925), server CPU 0 (Cortex-A725), path 10.201
  transport=True, parse=True, average_us=1.6
Trial 1, client CPU 5 (Cortex-X925), server CPU 0 (Cortex-A725), path 10.200
  transport=True, parse=True, average_us=1.87
Trial 1, client CPU 5 (Cortex-X925), server CPU 5 (Cortex-X925), path 10.200
  transport=True, parse=True, average_us=1.86
Trial 1, client CPU 5 (Cortex-X925), server CPU 5 (Cortex-X925), path 10.201
  transport=True, parse=True, average_us=1.47
Trial 2, client C

In [52]:
display(
    asymmetric_latency_results_df[
        [
            "trial",
            "client_cpu",
            "client_core_type",
            "server_cpu",
            "server_core_type",
            "path",
            "typical_us",
            "average_us",
            "percentile_99_us",
            "percentile_99_9_us",
            "succeeded",
        ]
    ]
)

,trial,client_cpu,client_core_type,server_cpu,server_core_type,path,typical_us,average_us,percentile_99_us,percentile_99_9_us,succeeded
0,1,0,Cortex-A725,0,Cortex-A725,10.201,1.76,1.74,2.08,2.44,True
1,1,0,Cortex-A725,0,Cortex-A725,10.200,2.03,1.99,2.33,2.67,True
2,1,0,Cortex-A725,5,Cortex-X925,10.200,2.02,1.97,2.30,2.45,True
3,1,0,Cortex-A725,5,Cortex-X925,10.201,1.51,1.61,1.86,2.14,True
4,1,5,Cortex-X925,0,Cortex-A725,10.201,1.50,1.60,1.86,2.10,True
5,1,5,Cortex-X925,0,Cortex-A725,10.200,1.77,1.87,2.27,2.37,True
6,1,5,Cortex-X925,5,Cortex-X925,10.200,1.76,1.86,2.26,2.34,True
7,1,5,Cortex-X925,5,Cortex-X925,10.201,1.47,1.47,1.54,1.71,True
8,2,5,Cortex-X925,5,Cortex-X925,10.200,1.75,1.85,2.25,2.35,True
9,2,5,Cortex-X925,5,Cortex-X925,10.201,1.46,1.46,1.57,1.78,True


In [ ]:
asymmetric_latency_summary_df = asymmetric_latency_results_df.groupby(
    [
        "client_cpu",
        "client_core_type",
        "server_cpu",
        "server_core_type",
        "path",
    ],
    as_index=False,
).agg(
    trials=("trial", "count"),
    median_typical_us=("typical_us", "median"),
    median_average_us=("average_us", "median"),
    mean_average_us=("average_us", "mean"),
    median_99_us=("percentile_99_us", "median"),
    median_99_9_us=("percentile_99_9_us", "median"),
)

display(asymmetric_latency_summary_df)

,client_cpu,client_core_type,server_cpu,server_core_type,path,trials,median_typical_us,median_average_us,mean_average_us,median_99_us,median_99_9_us
0,0,Cortex-A725,0,Cortex-A725,10.200,3,2.03,1.99,1.993333,2.33,2.59
1,0,Cortex-A725,0,Cortex-A725,10.201,3,1.77,1.74,1.743333,2.09,2.44
2,0,Cortex-A725,5,Cortex-X925,10.200,3,2.02,1.97,1.966667,2.30,2.45
3,0,Cortex-A725,5,Cortex-X925,10.201,3,1.50,1.60,1.603333,1.86,2.14
4,5,Cortex-X925,0,Cortex-A725,10.200,3,1.77,1.88,1.880000,2.27,2.37
5,5,Cortex-X925,0,Cortex-A725,10.201,3,1.50,1.61,1.606667,1.86,2.10
6,5,Cortex-X925,5,Cortex-X925,10.200,3,1.75,1.85,1.853333,2.25,2.34
7,5,Cortex-X925,5,Cortex-X925,10.201,3,1.46,1.46,1.463333,1.57,1.73


In [54]:
asymmetric_effects_df = asymmetric_latency_summary_df.pivot_table(
    index=["client_cpu", "client_core_type"],
    columns=["server_cpu", "server_core_type", "path"],
    values="median_average_us",
)

display(asymmetric_effects_df)

server_cpu                            0                  5       
server_core_type            Cortex-A725        Cortex-X925       
path                             10.200 10.201      10.200 10.201
client_cpu client_core_type                                      
0          Cortex-A725             1.99   1.74        1.97   1.60
5          Cortex-X925             1.88   1.61        1.85   1.46

In [55]:
comparison_rows = []

for path in ("10.200", "10.201"):
    path_results = asymmetric_latency_summary_df[
        asymmetric_latency_summary_df["path"] == path
    ]

    def median_for(client_cpu: int, server_cpu: int) -> float:
        row = path_results[
            (path_results["client_cpu"] == client_cpu)
            & (path_results["server_cpu"] == server_cpu)
        ]
        return float(row["median_average_us"].iloc[0])

    aa = median_for(0, 0)
    ax = median_for(0, 5)
    xa = median_for(5, 0)
    xx = median_for(5, 5)

    comparison_rows.append(
        {
            "path": path,
            "A725_client_A725_server_us": aa,
            "A725_client_X925_server_us": ax,
            "X925_client_A725_server_us": xa,
            "X925_client_X925_server_us": xx,
            "server_upgrade_with_A725_client_us": ax - aa,
            "server_upgrade_with_X925_client_us": xx - xa,
            "client_upgrade_with_A725_server_us": xa - aa,
            "client_upgrade_with_X925_server_us": xx - ax,
        }
    )

asymmetric_effect_comparison_df = pd.DataFrame(comparison_rows)
display(asymmetric_effect_comparison_df)

,path,A725_client_A725_server_us,A725_client_X925_server_us,X925_client_A725_server_us,X925_client_X925_server_us,server_upgrade_with_A725_client_us,server_upgrade_with_X925_client_us,client_upgrade_with_A725_server_us,client_upgrade_with_X925_server_us
0,10.200,1.99,1.97,1.88,1.85,-0.02,-0.03,-0.11,-0.12
1,10.201,1.74,1.60,1.61,1.46,-0.14,-0.15,-0.13,-0.14


### Prediction: sustained RDMA write bandwidth

The active link reports a nominal rate of 200 Gb/s, but that value is physical
link metadata rather than measured payload throughput.

For a sufficiently large message, RDMA write bandwidth should be much higher
than the functional `ibv_rc_pingpong` result and should approach a substantial
fraction of the link rate. It may remain below 200 Gb/s because of protocol
overhead, PCIe and memory-system limits, queue depth, and benchmark parameters.

This first bandwidth run is a format and functionality smoke test. It uses one
path, one message size, and one queue pair. Its result is not yet a complete
bandwidth characterization.

In [56]:
bandwidth_help_result = run_command(
    ("ib_write_bw", "--help"),
    timeout=10.0,
)

help_lines = [
    line
    for line in (
        bandwidth_help_result.stdout + bandwidth_help_result.stderr
    ).splitlines()
    if any(
        keyword in line.lower()
        for keyword in (
            "duration",
            "size",
            "iters",
            "gid",
            "device",
            "report_gbits",
            "queue",
        )
    )
]

print("\n".join(help_lines))

  -a, --all  Run sizes from 2 till 2^23
      --aes_block_size=<512,520,4048,4096,4160> (default 512)
  -d, --ib-dev=<dev>  Use IB device <dev> (default first device found)
  -D, --duration  Run test for a customized period of seconds.
  -i, --ib-port=<port>  Use port <port> of IB device (default 1)
  -I, --inline_size=<size>  Max size of message to be sent in inline
  -l, --post_list=<list size>
 Post list of send WQEs of <list size> size (instead of single post)
      --recv_post_list=<list size> Post list of receive WQEs of <list size> size (instead of single post)
  -m, --mtu=<mtu>  MTU size : 256 - 4096 (default port mtu)
  -n, --iters=<iters>  Number of exchanges (at least 5, default 5000)
  -N, --noPeak Cancel peak-bw calculation (default with peak up to iters=20000)
  -s, --size=<size>  Size of message to exchange (default 65536)
  -t, --tx-depth=<dep>  Size of tx queue (default 128)
  -x, --gid-index=<index>  Test uses GID with GID index
      --cpu_util  Show CPU Utilization 

In [57]:
BANDWIDTH_PATH = {
    "path": "10.201",
    "peer_address": "10.201.0.2",
    "rdma_device": "rocep1s0f1",
}

BANDWIDTH_CPU = 15
BANDWIDTH_MESSAGE_BYTES = 65_536
BANDWIDTH_DURATION_SECONDS = 5
BANDWIDTH_CONTROL_PORT = 19_500

In [ ]:
bandwidth_arguments = (
    "ib_write_bw",
    "-d",
    BANDWIDTH_PATH["rdma_device"],
    "-i",
    str(LATENCY_PORT),
    "-x",
    str(LATENCY_GID_INDEX),
    "-s",
    str(BANDWIDTH_MESSAGE_BYTES),
    "-D",
    str(BANDWIDTH_DURATION_SECONDS),
    "-p",
    str(BANDWIDTH_CONTROL_PORT),
    "-F",
    "--report_gbits",
)

bandwidth_server_command = (
    "ssh",
    "-T",
    "-o",
    "BatchMode=yes",
    "-o",
    "ConnectTimeout=5",
    peer_host,
    "timeout",
    "20s",
    "taskset",
    "-c",
    str(BANDWIDTH_CPU),
    *bandwidth_arguments,
)

bandwidth_client_command = (
    "timeout",
    "15s",
    "taskset",
    "-c",
    str(BANDWIDTH_CPU),
    *bandwidth_arguments,
    BANDWIDTH_PATH["peer_address"],
)

print("$ " + " ".join(bandwidth_server_command))

bandwidth_server_process = subprocess.Popen(
    bandwidth_server_command,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
)

try:
    time.sleep(1.0)

    bandwidth_client_result = run_command(
        bandwidth_client_command,
        timeout=20.0,
    )

    try:
        (
            bandwidth_server_stdout,
            bandwidth_server_stderr,
        ) = bandwidth_server_process.communicate(timeout=10.0)
    except subprocess.TimeoutExpired:
        bandwidth_server_process.terminate()

        try:
            (
                bandwidth_server_stdout,
                bandwidth_server_stderr,
            ) = bandwidth_server_process.communicate(timeout=5.0)
        except subprocess.TimeoutExpired:
            bandwidth_server_process.kill()
            (
                bandwidth_server_stdout,
                bandwidth_server_stderr,
            ) = bandwidth_server_process.communicate()
finally:
    if bandwidth_server_process.poll() is None:
        bandwidth_server_process.terminate()

        try:
            bandwidth_server_process.wait(timeout=5.0)
        except subprocess.TimeoutExpired:
            bandwidth_server_process.kill()
            bandwidth_server_process.wait()

print_command_result(bandwidth_client_result)

print("Remote server result")
print(f"return code: {bandwidth_server_process.returncode}")

if bandwidth_server_stdout.strip():
    print("\nstdout:")
    print(bandwidth_server_stdout.rstrip())

if bandwidth_server_stderr.strip():
    print("\nstderr:")
    print(bandwidth_server_stderr.rstrip())

bandwidth_smoke_succeeded = (
    bandwidth_client_result.succeeded and bandwidth_server_process.returncode == 0
)

print(f"\nBandwidth smoke test succeeded: {bandwidth_smoke_succeeded}")

$ ssh -T -o BatchMode=yes -o ConnectTimeout=5 spark-f868 timeout 20s taskset -c 15 ib_write_bw -d rocep1s0f1 -i 1 -x 3 -s 65536 -D 5 -p 19500 -F --report_gbits
$ timeout 15s taskset -c 15 ib_write_bw -d rocep1s0f1 -i 1 -x 3 -s 65536 -D 5 -p 19500 -F --report_gbits 10.201.0.2
return code: 0
status: succeeded

stdout:
---------------------------------------------------------------------------------------
                    RDMA_Write BW Test
 Dual-port       : OFF		Device         : rocep1s0f1
 Number of qps   : 1		Transport type : IB
 Connection type : RC		Using SRQ      : OFF
 PCIe relax order: ON
 ibv_wr* API     : ON
 TX depth        : 128
 CQ Moderation   : 1
 Mtu             : 4096[B]
 Link type       : Ethernet
 GID index       : 3
 Max inline data : 0[B]
 rdma_cm QPs	 : OFF
 Data ex. method : Ethernet
---------------------------------------------------------------------------------------
 local address: LID 0000 QPN 0x025c PSN 0xaaa2be RKey 0x1befed VAddr 0x00e0a1f793d000
 GID: 0

In [59]:
bandwidth_smoke_summary_df = pd.DataFrame(
    [
        {
            "path": BANDWIDTH_PATH["path"],
            "rdma_device": BANDWIDTH_PATH["rdma_device"],
            "message_bytes": BANDWIDTH_MESSAGE_BYTES,
            "duration_seconds": BANDWIDTH_DURATION_SECONDS,
            "cpu": BANDWIDTH_CPU,
            "client_returncode": bandwidth_client_result.returncode,
            "server_returncode": bandwidth_server_process.returncode,
            "succeeded": bandwidth_smoke_succeeded,
        }
    ]
)

display(bandwidth_smoke_summary_df)

,path,rdma_device,message_bytes,duration_seconds,cpu,client_returncode,server_returncode,succeeded
0,10.201,rocep1s0f1,65536,5,15,0,0,True


### Prediction: message size and RDMA write bandwidth

Small writes should be limited by the number of work requests and completions
that one CPU and queue pair can process. As message size increases, byte
throughput should rise until the link, PCIe path, memory system, or benchmark
implementation becomes the dominant bottleneck.

I predict:

1. message rate will be highest for small messages but byte throughput will be
   low;
2. bandwidth will increase rapidly from tens of bytes through kilobyte-scale
   messages;
3. large-message bandwidth will approach a plateau;
4. the two PCI paths may show different saturation bandwidth, even though both
   report the same nominal 200 Gb/s link rate;
5. the latency advantage previously observed on `10.201` does not necessarily
   imply that it will also have higher large-message bandwidth.

In [60]:
BANDWIDTH_MESSAGE_SIZES = (
    64,
    256,
    1_024,
    4_096,
    16_384,
    65_536,
    262_144,
    1_048_576,
)

BANDWIDTH_TRIALS = 3
BANDWIDTH_DURATION_SECONDS = 3
BANDWIDTH_CPU = 15

In [61]:
def parse_perftest_bandwidth(
    output: str,
) -> dict[str, float | int] | None:
    rows = []

    for line in output.splitlines():
        fields = line.split()

        if len(fields) != 5:
            continue

        try:
            message_bytes = int(fields[0])
            iterations = int(fields[1])
            peak_gbps = float(fields[2])
            average_gbps = float(fields[3])
            message_rate_mpps = float(fields[4])
        except ValueError:
            continue

        rows.append(
            {
                "message_bytes": message_bytes,
                "iterations": iterations,
                "peak_gbps": peak_gbps,
                "average_gbps": average_gbps,
                "message_rate_mpps": message_rate_mpps,
            }
        )

    return rows[-1] if rows else None

In [ ]:
def run_write_bandwidth_trial(
    *,
    trial: int,
    path: dict[str, str],
    message_bytes: int,
    control_port: int,
) -> dict[str, object]:
    benchmark_arguments = (
        "ib_write_bw",
        "-d",
        path["rdma_device"],
        "-i",
        str(LATENCY_PORT),
        "-x",
        str(LATENCY_GID_INDEX),
        "-s",
        str(message_bytes),
        "-D",
        str(BANDWIDTH_DURATION_SECONDS),
        "-p",
        str(control_port),
        "-F",
        "--report_gbits",
    )

    server_command = (
        "ssh",
        "-T",
        "-o",
        "BatchMode=yes",
        "-o",
        "ConnectTimeout=5",
        peer_host,
        "timeout",
        "15s",
        "taskset",
        "-c",
        str(BANDWIDTH_CPU),
        *benchmark_arguments,
    )

    client_command = (
        "timeout",
        "12s",
        "taskset",
        "-c",
        str(BANDWIDTH_CPU),
        *benchmark_arguments,
        path["peer_address"],
    )

    server_process = subprocess.Popen(
        server_command,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
    )

    client_result = None
    server_stdout = ""
    server_stderr = ""

    try:
        time.sleep(1.0)
        client_result = run_command(client_command, timeout=15.0)

        try:
            server_stdout, server_stderr = server_process.communicate(timeout=8.0)
        except subprocess.TimeoutExpired:
            server_process.terminate()

            try:
                server_stdout, server_stderr = server_process.communicate(timeout=4.0)
            except subprocess.TimeoutExpired:
                server_process.kill()
                server_stdout, server_stderr = server_process.communicate()
    finally:
        if server_process.poll() is None:
            server_process.terminate()

            try:
                server_process.wait(timeout=4.0)
            except subprocess.TimeoutExpired:
                server_process.kill()
                server_process.wait()

    parsed = (
        parse_perftest_bandwidth(client_result.stdout)
        if client_result is not None
        else None
    )

    transport_succeeded = (
        client_result is not None
        and client_result.succeeded
        and server_process.returncode == 0
    )
    parse_succeeded = parsed is not None

    return {
        "trial": trial,
        **path,
        "requested_message_bytes": message_bytes,
        "control_port": control_port,
        "client_returncode": (
            client_result.returncode if client_result is not None else None
        ),
        "server_returncode": server_process.returncode,
        "transport_succeeded": transport_succeeded,
        "parse_succeeded": parse_succeeded,
        "succeeded": transport_succeeded and parse_succeeded,
        "client_stdout": (client_result.stdout if client_result is not None else ""),
        "client_stderr": (client_result.stderr if client_result is not None else ""),
        "server_stdout": server_stdout,
        "server_stderr": server_stderr,
        **(parsed or {}),
    }

In [ ]:
bandwidth_measurements = []
measurement_index = 0

for trial in range(1, BANDWIDTH_TRIALS + 1):
    ordered_sizes = (
        BANDWIDTH_MESSAGE_SIZES
        if trial % 2 == 1
        else tuple(reversed(BANDWIDTH_MESSAGE_SIZES))
    )

    for size_index, message_bytes in enumerate(ordered_sizes):
        ordered_paths = (
            latency_paths
            if (trial + size_index) % 2 == 0
            else tuple(reversed(latency_paths))
        )

        for path in ordered_paths:
            measurement_index += 1
            control_port = 19_600 + measurement_index

            print(f"Trial {trial}, size {message_bytes} bytes, path {path['path']}")

            measurement = run_write_bandwidth_trial(
                trial=trial,
                path=path,
                message_bytes=message_bytes,
                control_port=control_port,
            )
            bandwidth_measurements.append(measurement)

            print(
                f"  transport={measurement['transport_succeeded']}, "
                f"parse={measurement['parse_succeeded']}, "
                f"average_gbps={measurement.get('average_gbps')}, "
                f"message_rate_mpps="
                f"{measurement.get('message_rate_mpps')}"
            )

            if not measurement["succeeded"]:
                print("\nClient stdout:")
                print(measurement["client_stdout"])
                print("\nClient stderr:")
                print(measurement["client_stderr"])
                print("\nServer stdout:")
                print(measurement["server_stdout"])
                print("\nServer stderr:")
                print(measurement["server_stderr"])
                raise RuntimeError(
                    f"Bandwidth measurement failed for "
                    f"{message_bytes} bytes on path {path['path']}"
                )

bandwidth_results_df = pd.DataFrame(bandwidth_measurements)

Trial 1, size 64 bytes, path 10.201
  transport=True, parse=True, average_gbps=5.56, message_rate_mpps=10.865457
Trial 1, size 64 bytes, path 10.200
  transport=True, parse=True, average_gbps=5.51, message_rate_mpps=10.755186
Trial 1, size 256 bytes, path 10.200
  transport=True, parse=True, average_gbps=21.14, message_rate_mpps=10.322602
Trial 1, size 256 bytes, path 10.201
  transport=True, parse=True, average_gbps=21.36, message_rate_mpps=10.430009
Trial 1, size 1024 bytes, path 10.201
  transport=True, parse=True, average_gbps=65.89, message_rate_mpps=8.043373
Trial 1, size 1024 bytes, path 10.200
  transport=True, parse=True, average_gbps=65.19, message_rate_mpps=7.957517
Trial 1, size 4096 bytes, path 10.200
  transport=True, parse=True, average_gbps=103.51, message_rate_mpps=3.158788
Trial 1, size 4096 bytes, path 10.201
  transport=True, parse=True, average_gbps=103.72, message_rate_mpps=3.165293
Trial 1, size 16384 bytes, path 10.201
  transport=True, parse=True, average_gbps=

In [64]:
display(
    bandwidth_results_df[
        [
            "trial",
            "path",
            "rdma_device",
            "message_bytes",
            "iterations",
            "peak_gbps",
            "average_gbps",
            "message_rate_mpps",
            "succeeded",
        ]
    ]
)

,trial,path,rdma_device,message_bytes,iterations,peak_gbps,average_gbps,message_rate_mpps,succeeded
0,1,10.201,rocep1s0f1,64,32596100,0.0,5.56,10.865457,True
1,1,10.200,roceP2p1s0f1,64,32265400,0.0,5.51,10.755186,True
2,1,10.200,roceP2p1s0f1,256,30967900,0.0,21.14,10.322602,True
3,1,10.201,rocep1s0f1,256,31290100,0.0,21.36,10.430009,True
4,1,10.201,rocep1s0f1,1024,24130100,0.0,65.89,8.043373,True
5,1,10.200,roceP2p1s0f1,1024,23872400,0.0,65.19,7.957517,True
6,1,10.200,roceP2p1s0f1,4096,9476400,0.0,103.51,3.158788,True
7,1,10.201,rocep1s0f1,4096,9495800,0.0,103.72,3.165293,True
8,1,10.201,rocep1s0f1,16384,2482104,0.0,108.45,0.827370,True
9,1,10.200,roceP2p1s0f1,16384,2481028,0.0,108.40,0.827009,True


In [ ]:
bandwidth_summary_df = bandwidth_results_df.groupby(
    ["path", "message_bytes"],
    as_index=False,
).agg(
    trials=("trial", "count"),
    median_average_gbps=("average_gbps", "median"),
    mean_average_gbps=("average_gbps", "mean"),
    minimum_average_gbps=("average_gbps", "min"),
    maximum_average_gbps=("average_gbps", "max"),
    median_message_rate_mpps=("message_rate_mpps", "median"),
)

bandwidth_summary_df["median_gbytes_per_second"] = (
    bandwidth_summary_df["median_average_gbps"] / 8
)
bandwidth_summary_df["nominal_link_utilization"] = (
    bandwidth_summary_df["median_average_gbps"] / 200
)

display(bandwidth_summary_df)

,path,message_bytes,trials,median_average_gbps,mean_average_gbps,minimum_average_gbps,maximum_average_gbps,median_message_rate_mpps,median_gbytes_per_second,nominal_link_utilization
0,10.200,64,3,5.50,5.500000,5.49,5.51,10.737785,0.68750,0.02750
1,10.200,256,3,21.20,21.183333,21.14,21.21,10.352430,2.65000,0.10600
2,10.200,1024,3,66.17,66.070000,65.19,66.85,8.076863,8.27125,0.33085
3,10.200,4096,3,103.52,103.516667,103.51,103.52,3.159067,12.94000,0.51760
4,10.200,16384,3,108.40,108.393333,108.38,108.40,0.827009,13.55000,0.54200
5,10.200,65536,3,109.12,109.116667,109.07,109.16,0.208130,13.64000,0.54560
6,10.200,262144,3,109.19,109.206667,109.17,109.26,0.052064,13.64875,0.54595
7,10.200,1048576,3,109.28,109.270000,109.25,109.28,0.013028,13.66000,0.54640
8,10.201,64,3,5.56,5.556667,5.55,5.56,10.855506,0.69500,0.02780
9,10.201,256,3,21.51,21.506667,21.36,21.65,10.503786,2.68875,0.10755


In [ ]:
bandwidth_path_pairs_df = bandwidth_summary_df.pivot(
    index="message_bytes",
    columns="path",
    values="median_average_gbps",
).reset_index()

bandwidth_path_pairs_df["10.201_minus_10.200_gbps"] = (
    bandwidth_path_pairs_df["10.201"] - bandwidth_path_pairs_df["10.200"]
)
bandwidth_path_pairs_df["10.201_over_10.200"] = (
    bandwidth_path_pairs_df["10.201"] / bandwidth_path_pairs_df["10.200"]
)

display(bandwidth_path_pairs_df)

path,message_bytes,10.200,10.201,10.201_minus_10.200_gbps,10.201_over_10.200
0,64,5.50,5.56,0.06,1.010909
1,256,21.20,21.51,0.31,1.014623
2,1024,66.17,65.85,-0.32,0.995164
3,4096,103.52,103.77,0.25,1.002415
4,16384,108.40,108.45,0.05,1.000461
5,65536,109.12,109.20,0.08,1.000733
6,262144,109.19,109.27,0.08,1.000733
7,1048576,109.28,109.27,-0.01,0.999908


In [70]:
bandwidth_results_df["derived_gbps"] = (
    bandwidth_results_df["message_bytes"]
    * bandwidth_results_df["message_rate_mpps"]
    * 8
    / 1_000
)

bandwidth_results_df["throughput_rounding_residual_gbps"] = (
    bandwidth_results_df["average_gbps"] - bandwidth_results_df["derived_gbps"]
)

bandwidth_results_df["relative_difference_percent"] = (
    bandwidth_results_df["throughput_rounding_residual_gbps"]
    / bandwidth_results_df["average_gbps"]
    * 100
)

display(
    bandwidth_results_df[
        [
            "trial",
            "path",
            "message_bytes",
            "average_gbps",
            "derived_gbps",
            "throughput_rounding_residual_gbps",
            "relative_difference_percent",
        ]
    ]
)

,trial,path,message_bytes,average_gbps,derived_gbps,throughput_rounding_residual_gbps,relative_difference_percent
0,1,10.201,64,5.56,5.563114,-0.003114,-0.056007
1,1,10.200,64,5.51,5.506655,0.003345,0.060704
2,1,10.200,256,21.14,21.140689,-0.000689,-0.003259
3,1,10.201,256,21.36,21.360658,-0.000658,-0.003083
4,1,10.201,1024,65.89,65.891312,-0.001312,-0.001991
5,1,10.200,1024,65.19,65.187979,0.002021,0.003100
6,1,10.200,4096,103.51,103.507165,0.002835,0.002739
7,1,10.201,4096,103.72,103.720321,-0.000321,-0.000310
8,1,10.201,16384,108.45,108.445041,0.004959,0.004573
9,1,10.200,16384,108.40,108.397724,0.002276,0.002100


## Observations

Run the cells above on one DGX Spark and preserve their outputs before drawing conclusions.

The initial interpretation should answer:

1. Which RDMA utilities are installed?
2. Which RDMA devices are visible?
3. Which kernel driver and PCI device back each RDMA device?
4. Which Linux network interfaces map to those devices?
5. What link layer does each RDMA port report?
6. Which ports are active, and what nominal rate do they report?
7. Are the mapped network interfaces up and carrying an IP address?
8. Does the routing table contain routes through those interfaces?

No bandwidth, latency, CPU-overhead, or GPU-direct conclusions can be made from this inventory alone.

## Explanation

TODO: Relate the observed device and link information to the RDMA data path.

## Connection to LLMs

RDMA supports low-latency transfer of tensors and collective traffic in distributed inference and training.

## Further Exploration

TODO: Define a two-node bandwidth and latency experiment with explicit safety and cleanup steps.